# BME KG Test Population

This demo shows how the different components in the workflow are used to convert the annotated object definition into triples, following classification of cateogry and type, linking, and data lifting (aka triplification) based on the simplified schema. The demo uses a test CSV (a subset of the whole collection). 

In [ ]:
from pathlib import Path
from populate import ingest_data_from_csv
from metamodel import LinkedEntity

testset_path = Path("../../assets/data/testset_flat_nt.csv")
GEMINI_API_KEY = "" # Replace with your actual API key

### Metamodel ingestion

In [12]:
# First of all we import the test collection from the CSV file and create Pydantic Item objects
objects = ingest_data_from_csv(testset_path)

Reading data from ../../assets/data/testset_flat_nt.csv...
Successfully ingested and created 65 Item objects.


/Users/jacopodeberardinis/Documents/projects/bme-kg/bmeapi/src/populate.py:163: UserWarning: CSV column 'item_model' not in metamodel and will be ignored.
  warnings.warn(
/Users/jacopodeberardinis/Documents/projects/bme-kg/bmeapi/src/populate.py:163: UserWarning: CSV column 'year' not in metamodel and will be ignored.
  warnings.warn(


In [14]:
# Let's show one of the objects
sample_object = objects[28]
sample_object

Item(name='Ziggy Stardust Wool Multi-coloured Suit', identifier='6eafffeb8a')

In [15]:
# Let's print all the fields of the object
for field, value in sample_object.model_dump().items():
    print(f"{field}: {value}")

identifier: 6eafffeb8a
name: Ziggy Stardust Wool Multi-coloured Suit
category: {'uri': None, 'label': 'Clothing'}
type: {'uri': None, 'label': 'suit'}
description: Ziggy Stardust Wool Multi-coloured Suit with feather boa
caption: 5 David Bowie Ziggy Stardust Costume 1973 This costume was designed by Kansai Yamamoto. His style, a contemporary take on traditional Japanese clothing, drawn from Samarui, Noh and Kabuki theatre and village styles was loved by Bowie, who commissioned him to produce nine costumes. These were the flamboyant outfits Bowie wore as Ziggy for the UK tour in 1973. “I love his music and obviously that has influenced my designs but most of all there's this aura of fantasy that surrounds him. He has flair" Kansai Yamamoto.
case_number: 10
exhibition: [{'uri': None, 'label': '4'}]
image: None
audio: None
video: None
genre: None
music_work: None
music_album: None
music_artist: [{'uri': None, 'label': 'David Bowie'}]
event: None
country: None
place: None
venue: None
colou

In [ ]:
# Now we can classify the object category and type
from classification import HierarchicalObjectClassifier  

classifier = HierarchicalObjectClassifier(
    generator_api_key=GEMINI_API_KEY,
    embedding_model='all-MiniLM-L6-v2',
    generator_model='gemini-2.5-flash'
)

### Category classification

In [ ]:
object_category = classifier.categoriser.classify(sample_object.description)

In [ ]:
object_category = object_category.strip()
print(f"Predicted category: {object_category}")
print(f"Expected category: {sample_object.category}")

In [ ]:
import json

mappings_path = Path("../mappings_min.json")

with open(mappings_path, 'r') as f:
    mappings = json.load(f)

In [ ]:
category_dict = mappings["category"][object_category.replace(" ", "")]
# rename _type_uri as uri and _description as label
category_dict['uri'] = category_dict.pop('_type_uri')[0]
category_dict['label'] = category_dict.pop('_description')
# set the corresponding item field to the matched category
sample_object.category = LinkedEntity.model_validate(category_dict)

In [ ]:
sample_object.get_category()

### Type classification

In [ ]:
object_category

In [ ]:
selected_indices, candidates = classifier.classify(sample_object.description,
                                                   object_category=object_category)

In [ ]:
object_type = candidates[selected_indices[0]]
object_type

In [ ]:
sample_object.type = LinkedEntity(uri=object_type['uri'], label=object_type['term'])

### (Entity) Linking

In [ ]:
from linker import RAGEntityLinker

linker = RAGEntityLinker(
    generator_model='gemini-2.5-flash',
    generator_api_key=GEMINI_API_KEY,
)

In [ ]:
sample_object

In [ ]:
# Define the fields from the pydantic Item to be linked through the entity linker
field_names = ["genre", "music_work","music_album","music_artist", "event",
               "country", "place", "venue", "colour", "material"]

# for each field, which contains an optional list of LinkedEntity (uri, label)
# we perform entity linking for those entities without a uri
for field_name in field_names:
    entities = getattr(sample_object, field_name)
    if entities is not None:
        for i, entity in enumerate(entities):
            if entity.uri is None:
                print(f"Linking entity '{entity.label}' ('{field_name}')...")
                linked_entity = linker.link(
                    search_term=entity.label,
                    context_query=sample_object.description,
                    entity_type=field_name,
                    top_k=10,
                    verbose=False,
                    validate=False,
                )
                if linked_entity is not None:
                    print(f"  Linked to: {linked_entity}")
                    uri = "http://www.wikidata.org/entity/" \
                        + linked_entity["wikidata_id"]
                    entities[i] = LinkedEntity(
                        uri=uri, label=linked_entity["label"])
                else:
                    print(f"  No link found for: {entity.label}")
        # Update the field with the linked entities
        setattr(sample_object, field_name, entities)
        # print(f"Updated field '{field_name}': {getattr(sample_object, field_name)}")

In [ ]:
# Let's print all the fields of the object
for field, value in sample_object.model_dump().items():
    print(f"{field}: {value}")

In [ ]:
# Finally, we rename the exhibition label to ZoneLabel
for i, exhibition in enumerate(sample_object.exhibition):
    exhibition_label = exhibition.label
    if exhibition_label is not None and "Zone" not in exhibition_label:
        sample_object.exhibition[i].label = "Zone" + exhibition_label

### Triplification

In [ ]:
from scho_triplifier import Triplifier

uuid_triplifier = Triplifier(
    mappings_path=mappings_path,
    base_uri="http://radio.liv.ac.uk/bme/",
    uri_strategy='uuid'
)

In [ ]:
uuid_output_file = "sample_output_graph_uuid.ttl"
uuid_rdf_graph = uuid_triplifier.triplify(
    sample_object, output_path=uuid_output_file)

## Running on the testset

In [ ]:
from scho_triplifier import Triplifier

print("Initialising services (Classifier, Linker, Triplifier)...")
# Initialising the classifier (with both category and type models)
classifier = HierarchicalObjectClassifier(
    generator_api_key=GEMINI_API_KEY
)
# Initialising the entity linker for the entities to be linked
linker = RAGEntityLinker(
    generator_api_key=GEMINI_API_KEY
)
# Initialising the triplifier with mappings and URI strategy
triplifier = Triplifier(
    mappings_path=mappings_path,
    base_uri="http://radio.liv.ac.uk/bme/",
    uri_strategy='uuid',
)

# Run Pipeline on the items
items_to_process = ingest_data_from_csv(testset_path)
print(f"Found {len(items_to_process)} items to process.")

In [ ]:
items_to_process = items_to_process[:5]

In [ ]:
len(items_to_process)

In [ ]:
for item in items_to_process:
    print(item.name or item.identifier)

In [ ]:
item = items_to_process[1]
item.name or item.identifier

In [ ]:
# Category classification
cat_label = classifier.categoriser.classify(item.description).strip()
cat_key = cat_label.replace(" ", "")
if cat_key in mappings["category"]:
    cat_dict = mappings["category"][cat_key]
    item.category = LinkedEntity(
        uri=cat_dict['_type_uri'][0],
        label=cat_dict['_description']
    )
    print(f"  Category classified as: {item.category.label}")

In [ ]:
# Type classification (depends on category)
indices, candidates = classifier.classify(
    item.description, object_category=cat_label, top_k=20)
if indices:
    obj_type = candidates[indices[0]]
    item.type = LinkedEntity(
        uri=obj_type['uri'],
        label=obj_type['term']
    )
    print(f"  Type classified as: {item.type.label}")

In [ ]:
candidates

In [ ]:
item.type

In [ ]:
# print the name of the the selected items

classifier.classify(item.description, object_category=cat_label)

## From a JSON structure

In [2]:
import json
from populate import process_dict_row

json_path = "../../assets/data/dataset_enriched.json"
with open(json_path, 'r') as f:
    data = json.load(f)
    
print(f"Loaded {len(data)} items from JSON.")

Loaded 62 items from JSON.


In [3]:
sample_object = data[23]

In [4]:

def rename_keys(obj):
    renamed = {}
    key_map = {
        'base_type': 'category',
        'specific_type': 'type',
        'music_genres': 'genre',
        'music_works': 'music_work',
        'music_albums': 'music_album',
        'music_artists': 'music_artist',
        'events': 'event',
        'countries': 'country',
        'places': 'place',
        'venues': 'venue',
        'colours': 'colour',
        'materials': 'material',
        'item_title': 'name',
        'item_description': 'description'
    }
    for k, v in obj.items():
        if k in ['years', 'cardinality']:
            continue
        if k in key_map:
            renamed[key_map[k]] = v
        elif k == 'input':
            if 'item_description' in v:
                renamed['description'] = v['item_description']
            if 'caption' in v:
                renamed['caption'] = v['caption']
        else:
            renamed[k] = v # keep other keys unchanged
    renamed['identifier'] = ""  # placeholder for identifier
    renamed['case'] = None # FIXME
    renamed['exhibition'] = [] # FIXME
    return renamed

In [5]:
sample_renamed = rename_keys(sample_object)
sample_renamed

{'category': 'Written Source',
 'type': 'Lyrics',
 'genre': [],
 'music_work': ['The Drowners',
  'Beautiful Ones',
  'So Young',
  'Lazy',
  'Animal Nitrate',
  'Metal Mickey'],
 'music_album': ['Suede', 'Coming Up'],
 'music_artist': ['Suede', 'Brett Anderson'],
 'event': [],
 'country': [],
 'place': [],
 'venue': [],
 'colour': [],
 'material': [],
 'name': 'Lyrics to ‘Beautiful Ones’, ‘So Young’, ‘The Drowners’ and ‘Lazy’',
 'description': 'Suede - Lyrics to ‘The Drowners’',
 'caption': '5 Suede Lyrics to ‘Beautiful Ones’, ‘So Young’, ‘The Drowners’ and ‘Lazy’ Suede catapulted onto the scene in 1993 with their debut album, Suede. The No.1 album included the singles ‘Animal Nitrate’, ‘Metal Mickey’, ‘So Young’ and ‘The Drowners’ – songs that seemed to both embrace and condemn decadent urban life. Despite a number of changes to the line-up, the band continued their success and chart-topping album Coming Up spawned five Top 10 singles including ‘Lazy’ and ‘Beautiful Ones’. By kind pe

In [6]:
item = process_dict_row(sample_renamed)

In [9]:
# Let's print all the fields of the object
for field, value in item.model_dump().items():
    print(f"{field}: {value}")

identifier: 82a8dc10ac
name: Lyrics to ‘Beautiful Ones’, ‘So Young’, ‘The Drowners’ and ‘Lazy’
category: {'uri': None, 'label': 'Written Source'}
type: {'uri': None, 'label': 'Lyrics'}
description: Suede - Lyrics to ‘The Drowners’
caption: 5 Suede Lyrics to ‘Beautiful Ones’, ‘So Young’, ‘The Drowners’ and ‘Lazy’ Suede catapulted onto the scene in 1993 with their debut album, Suede. The No.1 album included the singles ‘Animal Nitrate’, ‘Metal Mickey’, ‘So Young’ and ‘The Drowners’ – songs that seemed to both embrace and condemn decadent urban life. Despite a number of changes to the line-up, the band continued their success and chart-topping album Coming Up spawned five Top 10 singles including ‘Lazy’ and ‘Beautiful Ones’. By kind permission of Brett Anderson
case_number: None
exhibition: []
image: None
audio: None
video: None
genre: []
music_work: [{'uri': None, 'label': 'The Drowners'}, {'uri': None, 'label': 'Beautiful Ones'}, {'uri': None, 'label': 'So Young'}, {'uri': None, 'label'